# Figure for geohash grids

In [ ]:
import geopandas as gpd
import pandas as pd
import altair as alt
from vega_datasets import data
import sys
sys.path.append("../")
from geohashtree.geohash_func import geohash_to_bbox

In [ ]:
result = gpd.read_parquet("../data/overture/us_places_bkgh.parquet")

In [ ]:
lvl = 2
df = pd.DataFrame({'value':result['geohash'].str[:lvl].unique(),'level':lvl})

gg2 = gpd.GeoDataFrame(df,geometry=df['value'].map(geohash_to_bbox),crs='EPSG:4326')

lvl = 3
df = pd.DataFrame({'value':result['geohash'].str[:lvl].unique(),'level':lvl})

gg3 = gpd.GeoDataFrame(df,geometry=df['value'].map(geohash_to_bbox),crs='EPSG:4326')

lvl = 4
df = pd.DataFrame({'value':result['geohash'].str[:lvl].unique(),'level':lvl})

gg4 = gpd.GeoDataFrame(df,geometry=df['value'].map(geohash_to_bbox),crs='EPSG:4326')


gg3 = gg3[gg3.value.str.startswith('dn')]

gg4 = gg4[gg4.value.str.startswith('dnp')]

gg4['lon'] = gg4["geometry"].centroid.x
gg4['lat'] = gg4["geometry"].centroid.y

gg2['lon'] = gg2["geometry"].centroid.x
gg2['lat'] = gg2["geometry"].centroid.y
gg3['lon'] = gg3["geometry"].centroid.x
gg3['lat'] = gg3["geometry"].centroid.y

In [ ]:
prj_type = 'identity'
exclude_ghs = ['dk','9k','9s','dn','c0','9p','9n','f8','dx','dw','9u','dh']
states = alt.topo_feature(data.us_10m.url, feature='states')

cont_us = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).transform_filter(
    "datum.id != 15 && datum.id != 2 && datum.id != 72"
).project(
    prj_type,
    reflectY=True,
    #scale= 5,                          # Magnify
    #center= [100,0],
    #extent=[[-120, 30], [-100, 50]]
    #clipExtent= [[400, 200], [600, 300]],
    #translate=[-10,-30]
         ).properties(
    width=5000,
    height=3000,
    
).interactive()

# alt_data  = alt.InlineData(values = geohash_grids.to_json(), #geopandas to geojson
#                        # root object type is "FeatureCollection" but we need its features
#                        format = alt.DataFormat(property='features',type='json'))

# gh = alt.Chart(gg2.query('value not in @exclude_ghs')).mark_geoshape(filled=False
# ).encode(
#     #stroke='value:N',
#     tooltip=["value:N"],
# )
gh2 = alt.Chart(gg3.query('value != "dnp"')).mark_geoshape(filled=False
)
gh3 = alt.Chart(gg4).mark_geoshape(filled=False
)
# text = gh.mark_text(
#     align="left",
#     baseline="middle",
#     dx=3,
#     fontSize=20
# ).encode(
#     text="value",
#     longitude='lon',
#     latitude='lat'
# )
text2 = gh2.mark_text(
    align="left",
    dx=-10,
    fontSize=15
).encode(
    text="value",
    longitude='lon',
    latitude='lat'
)
text3 = gh3.mark_text(
    align="left",
    dx=-10,
    fontSize=10
).encode(
    text="value",
    longitude='lon',
    latitude='lat'
)

In [ ]:
(cont_us+gh3+gh2+text2+text3)